# Setting

## Library

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Matplotlib global settings
mpl.rcParams["axes.titlesize"] = 14
mpl.rcParams["axes.labelsize"] = 20
plt.rcParams['savefig.dpi'] = 500
plt.rc('font', family='serif')

In [3]:
# ML libraries
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

In [4]:
# Helper functions & model import
sys.path.append(os.path.join('..', 'src'))
from helper import makeSpecColors
from paths import *
from var import *
from sdtpy import *
from model import *

## Function

In [5]:
def select_uids_by_class(df, sample_number, class_col='Class', uid_col='uid', random_state=42):
    np.random.seed(random_state)
    uids_by_class = {}
    for c in df[class_col].unique():
        uids = df[df[class_col]==c][uid_col].unique()
        n_select = min(sample_number, len(uids))
        selected_uids = np.random.choice(uids, n_select, replace=False)
        uids_by_class[c] = set(selected_uids)
    return uids_by_class

In [6]:
def filter_by_selected_uids(df, uids_by_class, class_col='Class', uid_col='uid'):
    mask = np.zeros(len(df), dtype=bool)
    for c, uids in uids_by_class.items():
        mask |= ((df[class_col]==c) & (df[uid_col].isin(uids)))
    return df[mask]

In [7]:
from sklearn.model_selection import GroupShuffleSplit

def train_test_split_by_uid(df, test_size=0.2, class_col='Class', uid_col='uid', random_state=42):
    groups = df[uid_col]
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(df, df[class_col], groups))
    return df.iloc[train_idx], df.iloc[test_idx]

## Initial Setup

In [8]:
logtxt = ""

In [9]:
# Set experiment configs
random_state = 42
test_size = 0.2
device_type = "cpu" # or gpu
n_jobs = 5
#
test_name = "Rubin_7DT_LightGBM"
path_save = os.path.join(MODEL, test_name)
os.makedirs(path_save, exist_ok=True)
#

logtxt += "\nSet experiment configs\n"
# logtxt += f"test_name: {test_name}\n"
logtxt += f"random_state: {random_state}\n"
logtxt += f"test_size: {test_size}\n"
logtxt += f"device_type: {device_type}\n"
logtxt += f"n_jobs: {n_jobs}\n"
logtxt += f"path_save: {path_save}\n"
logtxt += "\n"


- Source to Consider

In [10]:
sources_to_consider = [
	"AGN", 
	"Ia", 
	"II", 
	"Ibc", 
	"LBV", 
	"TDE", 
	"Nova", 
	"M dwarf", 
	"CV",
	"SLSN",
]
logtxt += f"\nSources to consider: {sources_to_consider}\n"

In [11]:
path_data = os.path.join(FEATURE_BALANCED_DATA, 'features_rubin_7dt40.csv')

logtxt += f"\nBalanced Data Set\n"

# Data

In [12]:
# columns_to_use = list(data_dtype_dict.keys())

In [13]:
data = pd.read_csv(
    path_data,
    engine='c', 
    # usecols=columns_to_use,
    # dtype=data_dtype_dict,
)

data['uid'] = data['uid'].astype(str)
data['Class'] = data['Class'].astype(str)

uids = data['uid'].values
classes = data['Class'].values

print(f"Balanced Data: {len(data)}")

logtxt += f"Balanced Data: {len(data)}\n"

indx_type_to_consider = np.where(
	np.array([(data['Class'] == source) for source in sources_to_consider]).any(axis=0)
)

print(f"{len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}")
data = data.iloc[indx_type_to_consider[0]]

logtxt += f"Balanced: {len(sources_to_consider)} sources to consider: {len(indx_type_to_consider[0])}\n"
logtxt += "\n"


Balanced Data: 282753
10 sources to consider: 282753


In [14]:
import yaml
path_feature_config = os.path.join(CONFIG, "feature.yaml")

with open(path_feature_config, "r") as f:
    feature_config = yaml.safe_load(f)

feature_key = "Rubin+7DT_40"
all_features = feature_config[feature_key]['color']

In [15]:
color_7dt_features = feature_config["7DT_40"]["color"]

In [16]:
lsst_filters = ["u", "g", "r", "i", "z", "y"]

In [17]:
rubin_added_feature_dict = {}

for filte in lsst_filters:
    features = color_7dt_features.copy()
    for feature in all_features:
        if filte in feature:
            features.append(feature)
        else:
            pass
    rubin_added_feature_dict[filte] = features

# rubin_added_feature_dict

## Train & Test Sample

In [18]:
# - Split features/target
X = data.drop(columns=['Sample_ID', 'Class', 'uid'])
y = data['Class']
X.fillna(-99, inplace=True)

# - Split into train/test using GroupShuffleSplit by uid
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(X, y, groups=data['uid']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# - Label encode class for ML
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)
class_names = np.array([str(c) for c in label_encoder.inverse_transform(np.arange(len(label_encoder.classes_)))])
print("Balanced: Class mapping:", class_names)

# Tets\\sts
classifier_type = 'normal_class_classifier'
model_param_config = model_config[classifier_type][device_type]

import lightgbm as lgb
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)


Balanced: Class mapping: ['AGN' 'CV' 'II' 'Ia' 'Ibc' 'LBV' 'M dwarf' 'Nova' 'SLSN' 'TDE']


In [19]:
del data

In [20]:
import yaml

# YAML 파일 열기
with open(f"../model/LightGBM_Fine_Tune/best_params.yaml", "r") as f:
    best_params = yaml.safe_load(f)

# 출력
best_params['num_threads'] = 5
best_params

{'colsample_bytree': 0.8342131273421189,
 'learning_rate': 0.09939493808165245,
 'min_child_weight': 19,
 'n_estimators': 71,
 'num_leaves': 27,
 'num_threads': 5,
 'random_state': 42}

# Train and Evaluation

In [21]:
# filte = 'u'

# feature_to_use = rubin_added_feature_dict[filte]
# X_train_sub = X_train[feature_to_use]
# X_test_sub = X_test[feature_to_use]

In [22]:
from lightgbm import LGBMClassifier  # 예시로 LGBM 사용

# best_model = LGBMClassifier(**best_params)
# best_model.fit(
#     X_train_sub, y_train,
#     )
# y_pred = best_model.predict(X_test_sub)

In [23]:
import joblib  # for XGBoost and LightGBM

# # ---------------------
# # CatBoost 저장
# # ---------------------
# cat_model.save_model(os.path.join(path_save, "best_catboost_model.txt"))

# # ---------------------
# # XGBoost 저장
# # ---------------------
# xgb_model.save_model(os.path.join(path_save, "best_xgboost_model.json"))  # .json preferred over .txt

# # ---------------------
# # LightGBM 저장
# # ---------------------
# joblib.dump(best_model, os.path.join(path_save, f"lightgbm_7DT_Rubin_{filte}.pkl"))

In [24]:
from sklearn.metrics import classification_report
import pandas as pd

# report_dict = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)

# # 2. accuracy는 scalar → 딕셔너리로 따로 저장
# acc_value = report_dict.pop("accuracy")

# # 3. DataFrame으로 변환
# report_df = pd.DataFrame(report_dict).T  # class별 성능 포함

# # 4. accuracy row 추가 (f1-score 칸만 채움)
# report_df.loc["accuracy"] = [None, None, acc_value, report_df["support"].sum()]

# report_df.to_csv(os.path.join(path_save, f"classification_report_{filte}.csv"))

In [25]:
def plot_confusion_matrix():
    import seaborn as sns
    cm = confusion_matrix(y_test, y_pred)
    labels = label_encoder.classes_
    cm_percent = (cm / cm.sum(axis=1, keepdims=True)) * 100
    n_rows, n_cols = cm.shape
    combined_matrix = np.empty_like(cm, dtype=object)
    for i in range(n_rows):
        for j in range(n_cols):
            n = cm[i, j]
            p = cm_percent[i, j]
            combined_matrix[i, j] = f"{n:,}\n({p:.1f}%)"
    plt.figure(figsize=(12, 9))
    ax = sns.heatmap(cm_percent, annot=False, fmt="", cmap="Blues",
                        xticklabels=labels, yticklabels=labels,
                        cbar_kws={'label': '[%]'}, vmin=0, vmax=100)
    plt.xlabel("Predicted Label", fontsize=16)
    plt.ylabel("True Label", fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    # plt.title(__class__.__name__.replace("Experiment", ""))
    plt.tight_layout()
    thresh = 50
    for i in range(n_rows):
        for j in range(n_cols):
            value = cm_percent[i, j]
            text_color = "white" if value > thresh else "black"
            ax.text(j+0.5, i+0.5, combined_matrix[i, j], ha='center', va='center', color=text_color, fontsize=14)
    # plt.savefig(os.path.join(path_save, f"{__class__.__name__.lower()}_confusion_matrix.png"))

# plot_confusion_matrix()

# plt.savefig(os.path.join(path_save, f'confusion_matrix_{filte}.png'))

In [26]:
# cm = confusion_matrix(y_test, y_pred)
# df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
# df_cm

# df_cm.to_csv(os.path.join(path_save, f'confusion_matrix_{filte}.csv'))

In [27]:
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report

# # 설정
# class_labels = label_encoder.classes_
# groups = uids
# gkf = GroupKFold(n_splits=5)

# cv_macro_f1_scores = []
# cv_per_class_f1_scores = []

# # 저장 경로 생성
# os.makedirs(path_save, exist_ok=True)

# Perform Grouped K-Fold Cross-Validation
"""
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups=groups)):

    X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_train_cv, y_val_cv = y_encoded[train_idx], y_encoded[val_idx]

    model_cv = LGBMClassifier(**best_params)
    model_cv.fit(
        X_train_cv, y_train_cv,
        eval_set=[(X_val_cv, y_val_cv)],
        # verbose=False
    )

    y_pred = model_cv.predict(X_val_cv)

    # === 1. Macro F1 저장 ===
    macro_f1 = f1_score(y_val_cv, y_pred, average='macro')
    cv_macro_f1_scores.append({'fold': fold, 'f1_macro': macro_f1})

    # === 2. Per-class F1 저장 ===
    report = classification_report(y_val_cv, y_pred, output_dict=True, zero_division=0)
    per_class_f1 = {label: report[str(i)]['f1-score'] for i, label in enumerate(class_labels)}
    per_class_f1['fold'] = fold
    cv_per_class_f1_scores.append(per_class_f1)

    # === 3. 모델 저장 ===
    model_path = os.path.join(path_save, f"model_fold{fold}.pkl")
    joblib.dump(model_cv, model_path)

    # === 4. 리포트 출력 ===
    print(f"[Fold {fold}] f1_macro = {macro_f1:.4f}")

# === 5. 최종 결과 저장 ===
macro_f1_df = pd.DataFrame(cv_macro_f1_scores)
per_class_f1_df = pd.DataFrame(cv_per_class_f1_scores)

macro_f1_df.to_csv(os.path.join(path_save, f"cv_macro_f1_scores_{filte}.csv"), index=False)
per_class_f1_df.to_csv(os.path.join(path_save, f"cv_per_class_f1_scores_{filte}.csv"), index=False)"""


'\nfor fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups=groups)):\n\n    X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]\n    y_train_cv, y_val_cv = y_encoded[train_idx], y_encoded[val_idx]\n\n    model_cv = LGBMClassifier(**best_params)\n    model_cv.fit(\n        X_train_cv, y_train_cv,\n        eval_set=[(X_val_cv, y_val_cv)],\n        # verbose=False\n    )\n\n    y_pred = model_cv.predict(X_val_cv)\n\n    # === 1. Macro F1 저장 ===\n    macro_f1 = f1_score(y_val_cv, y_pred, average=\'macro\')\n    cv_macro_f1_scores.append({\'fold\': fold, \'f1_macro\': macro_f1})\n\n    # === 2. Per-class F1 저장 ===\n    report = classification_report(y_val_cv, y_pred, output_dict=True, zero_division=0)\n    per_class_f1 = {label: report[str(i)][\'f1-score\'] for i, label in enumerate(class_labels)}\n    per_class_f1[\'fold\'] = fold\n    cv_per_class_f1_scores.append(per_class_f1)\n\n    # === 3. 모델 저장 ===\n    model_path = os.path.join(path_save, f"model_fold{fold}.

# Iteration

In [31]:
test_name = "Rubin_7DT_LightGBM"
path_save = os.path.join(MODEL, test_name)
os.makedirs(path_save, exist_ok=True)

from lightgbm import LGBMClassifier  # 예시로 LGBM 사용
import joblib  # for XGBoost and LightGBM
from sklearn.metrics import classification_report
import pandas as pd
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report

# 저장 경로 생성
os.makedirs(path_save, exist_ok=True)

for filte in lsst_filters:
    print(f"FILTER: {filte}")

    df_cm_filename = os.path.join(path_save, f'confusion_matrix_{filte}.csv')
    
    feature_to_use = rubin_added_feature_dict[filte]
    X_sub = X[feature_to_use]
    X_train_sub = X_train[feature_to_use]
    X_test_sub = X_test[feature_to_use]

    if not os.path.exists(df_cm_filename):
        
        #
        print(f"Start Training")
        best_model = LGBMClassifier(**best_params, force_col_wise=True)
        best_model.fit(
            X_train_sub, y_train,
            )
        y_pred = best_model.predict(X_test_sub)
        # best_model.save_model(f"{path_save}/best_lightgbm_model.txt")
        #
        
        # # ---------------------
        # # CatBoost 저장
        # # ---------------------
        # cat_model.save_model(os.path.join(path_save, "best_catboost_model.txt"))
        
        # # ---------------------
        # # XGBoost 저장
        # # ---------------------
        # xgb_model.save_model(os.path.join(path_save, "best_xgboost_model.json"))  # .json preferred over .txt
        
        # # ---------------------
        # # LightGBM 저장
        # # ---------------------
        joblib.dump(best_model, os.path.join(path_save, f"lightgbm_7DT_Rubin_{filte}.pkl"))
        #
    
        print(f"Get Classification Report")
        report_dict = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
        
        # 2. accuracy는 scalar → 딕셔너리로 따로 저장
        acc_value = report_dict.pop("accuracy")
        
        # 3. DataFrame으로 변환
        report_df = pd.DataFrame(report_dict).T  # class별 성능 포함
        
        # 4. accuracy row 추가 (f1-score 칸만 채움)
        report_df.loc["accuracy"] = [None, None, acc_value, report_df["support"].sum()]
        report_df.to_csv(os.path.join(path_save, f"classification_report_{filte}.csv"))
    
        print(f"Get Confusion Matrix")
        plot_confusion_matrix()
        plt.savefig(os.path.join(path_save, f'confusion_matrix_{filte}.png'))
        plt.close()
        
        cm = confusion_matrix(y_test, y_pred)
        df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
        df_cm.to_csv(df_cm_filename)
    else:
        print(df_cm_filename, "exists")
    
    
    # 설정
    class_labels = label_encoder.classes_
    groups = uids
    gkf = GroupKFold(n_splits=5)
    
    cv_macro_f1_scores = []
    cv_per_class_f1_scores = []
    

    print(f"Calculate 5-Fold Cross-Validation Values")

    # Perform Grouped K-Fold Cross-Validation
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y_encoded, groups=groups)):

        macro_f1_df_filename = os.path.join(path_save, f"cv_macro_f1_scores_{filte}_{fold}.csv")
        per_class_f1_df_filename = os.path.join(path_save, f"cv_per_class_f1_scores_{filte}_{fold}.csv")

        if (not os.path.exists(macro_f1_df_filename)) & ((not os.path.exists(per_class_f1_df_filename))):
            
            X_train_cv, X_val_cv = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
            y_train_cv, y_val_cv = y_encoded[train_idx], y_encoded[val_idx]
        
            model_cv = LGBMClassifier(**best_params, force_col_wise=True,)
            model_cv.fit(
                X_train_cv, y_train_cv,
                eval_set=[(X_val_cv, y_val_cv)],
                # verbose=False
            )
        
            y_pred = model_cv.predict(X_val_cv)
        
            # === 1. Macro F1 저장 ===
            macro_f1 = f1_score(y_val_cv, y_pred, average='macro')
            cv_macro_f1_scores.append({'fold': fold, 'f1_macro': macro_f1})
        
            # === 2. Per-class F1 저장 ===
            report = classification_report(y_val_cv, y_pred, output_dict=True, zero_division=0)
            per_class_f1 = {label: report[str(i)]['f1-score'] for i, label in enumerate(class_labels)}
            per_class_f1['fold'] = fold
            cv_per_class_f1_scores.append(per_class_f1)
        
            # === 3. 모델 저장 ===
            model_path = os.path.join(path_save, f"model_fold{fold}.pkl")
            joblib.dump(model_cv, model_path)
        
            # === 4. 리포트 출력 ===
            print(f"[Fold {fold}] f1_macro = {macro_f1:.3f}")
            
            # === 5. 최종 결과 저장 ===
            macro_f1_df = pd.DataFrame(cv_macro_f1_scores)
            per_class_f1_df = pd.DataFrame(cv_per_class_f1_scores)
    
    
            macro_f1_df.to_csv(macro_f1_df_filename, index=False)
            per_class_f1_df.to_csv(per_class_f1_df_filename, index=False)
        else:
            print(macro_f1_df_filename, "exists")
            print(per_class_f1_df_filename, "exists")


FILTER: u
/home/gpaek/SED-Classifier/notebook/../model/Rubin_7DT_LightGBM/confusion_matrix_u.csv exists
Calculate 5-Fold Cross-Validation Values
[LightGBM] [Info] Total Bins 209100
[LightGBM] [Info] Number of data points in the train set: 226202, number of used features: 820
[LightGBM] [Info] Start training from score -2.270960
[LightGBM] [Info] Start training from score -2.279563
[LightGBM] [Info] Start training from score -2.184791
[LightGBM] [Info] Start training from score -2.206681
[LightGBM] [Info] Start training from score -2.291865
[LightGBM] [Info] Start training from score -2.224267
[LightGBM] [Info] Start training from score -2.558713
[LightGBM] [Info] Start training from score -2.323907
[LightGBM] [Info] Start training from score -2.405943
[LightGBM] [Info] Start training from score -2.331068
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further sp